In [1]:
import torch
import tifffile
import numpy as np
import os
from typing import Optional, Tuple, Union
from tqdm import tqdm
from skimage import exposure
from micro_sam.automatic_segmentation import get_predictor_and_segmenter, automatic_instance_segmentation
from scipy.ndimage import label as ndimage_label, generate_binary_structure
from skimage.morphology import remove_small_objects


def run_automatic_instance_segmentation(
    image: np.ndarray,
    checkpoint_path: Union[os.PathLike, str],
    model_type: str = "vit_b_lm",
    device: Optional[Union[str, torch.device]] = None,
    tile_shape: Optional[Tuple[int, int]] = None,
    halo: Optional[Tuple[int, int]] = None,
):
    """Automatic Instance Segmentation (AIS) wrapper for convenience.
    
    Args:
        image: The input image (numpy array).
        checkpoint_path: The path to stored checkpoints.
        model_type: The choice of the µsam model.
        device: The device to run the model inference.
        tile_shape: The tile shape for tiling-based segmentation.
        halo: The overlap shape on each side per tile for stitching tiles.
    
    Returns:
        The instance segmentation result.
    """
    # Get the predictor and segmenter
    predictor, segmenter = get_predictor_and_segmenter(
        model_type=model_type,
        checkpoint=checkpoint_path,
        device=device,
        is_tiled=(tile_shape is not None),
    )
    
    # Run automatic instance segmentation
    prediction = automatic_instance_segmentation(
        predictor=predictor,
        segmenter=segmenter,
        input_path=image,
        ndim=2,
        tile_shape=tile_shape,
        halo=halo,
    )
    
    return prediction

In [12]:
# ============================
# 1. CONFIGURATION
# ============================
MODEL_TYPE = "vit_b_lm"
CHECKPOINT_PATH =  r'/work/scratch/rawat/models/microsam_training_output/checkpoints/vit_b_lm_patch1024_loss_ce/best.pt'
INPUT_IMAGE_PATH = r"/home/students/rawat/Dataset/Surface_Masks/Surface_Masks/Brain1_1340_Cont_AiryscanSR_Fly1_CalyxL.tif"
OUTPUT_MASK_PATH = r"/home/students/rawat/czi_output/Annotations/Processed_labels/pred_test/Brain1_1340_Cont_AiryscanSR_Fly1_CalyxL_tiling_896.tif"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Blob removal threshold
BLOB_REMOVAL_THRESHOLD = 1450

print(f"Using device: {DEVICE}")
print(f"Using model: {MODEL_TYPE} with checkpoint: {CHECKPOINT_PATH}")
print(f"Input image: {INPUT_IMAGE_PATH}")
print(f"Output will be saved to: {OUTPUT_MASK_PATH}")
print(f"Blob removal threshold: {BLOB_REMOVAL_THRESHOLD}")

Using device: cuda
Using model: vit_b_lm with checkpoint: /work/scratch/rawat/models/microsam_training_output/checkpoints/vit_b_lm_patch1024_loss_ce/best.pt
Input image: /home/students/rawat/Dataset/Surface_Masks/Surface_Masks/Brain1_1340_Cont_AiryscanSR_Fly1_CalyxL.tif
Output will be saved to: /home/students/rawat/czi_output/Annotations/Processed_labels/pred_test/Brain1_1340_Cont_AiryscanSR_Fly1_CalyxL_tiling_896.tif
Blob removal threshold: 1450


In [13]:
# ============================
# 2. LOAD AND PREPROCESS IMAGE
# ============================
print("Loading image...")
raw_data = tifffile.imread(INPUT_IMAGE_PATH)
print(f"Original shape: {raw_data.shape}")

# Extract Channel 0 based on axis order
if raw_data.ndim == 4:
    if raw_data.shape[0] <= 4:  # (C, Z, Y, X)
        raw_data = np.transpose(raw_data, (1, 2, 3, 0))
    elif raw_data.shape[1] <= 4:  # (Z, C, Y, X)
        raw_data = np.transpose(raw_data, (0, 2, 3, 1))
elif raw_data.ndim == 3:
        # Assume (Z, Y, X) if 3D, add dummy channel dim -> (Z, Y, X, 1)
        raw_data = raw_data[..., np.newaxis]

print(f"Processed shape (Z, Y, X, C): {raw_data.shape}")

print(f"The data type is {raw_data.dtype}, normalizing to uint8.")
# Calculate percentiles on the entire 3D volume at once
low, high = np.percentile(raw_data, (0.00001, 99.9999))
print(f"Percentile values: low={low}, high={high}")

if high > low:
    # Normalize to 0-1 as float32
    raw_data = (raw_data.astype(np.float32) - low) / (high - low)
    raw_data = np.clip(raw_data, 0, 1)
else:
    # Flat image fallback
    raw_data = np.zeros_like(raw_data, dtype=np.float32)

Z, Y, X, C = raw_data.shape
raw_data_rgb = np.zeros((Z, Y, X, 3), dtype=raw_data.dtype)

raw_data_rgb[..., :min(C, 2)] = raw_data[..., :min(C, 2)]
print(f"Final RGB shape: {raw_data_rgb.shape}")
print(f"Image preprocessing complete. Ready for inference.")

Loading image...
Original shape: (4, 70, 4, 1834, 1834)
Processed shape (Z, Y, X, C): (4, 70, 4, 1834, 1834)
The data type is uint8, normalizing to uint8.
Percentile values: low=0.0, high=255.0


ValueError: too many values to unpack (expected 4)

In [9]:
# ============================
# 3. INITIALIZE MODEL & RUN INFERENCE
# ============================
print("Loading model (will be reloaded per slice, which is necessary for proper tiling)...")

# ============================
# 4. RUN INFERENCE LOOP (Slice by Slice)
# ============================
prediction_stack = []

print(f"Starting Slice-by-Slice Inference on {raw_data_rgb.shape[0]} slices...")

# Tqdm gives you a progress bar
for z in tqdm(range(raw_data_rgb.shape[0]), desc="Processing Slices"):
    
    # A. Get single slice
    img_slice = raw_data_rgb[z] # (Y, X, C)

    # B. Predict using the wrapper function from microsam docs
    # This properly handles tiling with the predictor and segmenter
    pred_2d = run_automatic_instance_segmentation(
        image=img_slice,
        checkpoint_path=CHECKPOINT_PATH,
        model_type=MODEL_TYPE,
        device=DEVICE,
        tile_shape=(896, 896),  # 896 + 64 + 64 = 1024 (training patch size)
        halo=(64, 64)  # Overlap between tiles
    )
    
    prediction_stack.append(pred_2d)

# Stack results into 3D volume
print("Stacking inference results...")
predictions_3d = np.array(prediction_stack)
print(f"Prediction shape: {predictions_3d.shape}")
print(f"Inference complete. Output from this cell will be used for label connection.")

Loading model (will be reloaded per slice, which is necessary for proper tiling)...
Starting Slice-by-Slice Inference on 75 slices...


Compute Image Embeddings 2D tiled: 100%|██████████| 4/4 [00:01<00:00,  2.32it/s]
Initialize tiled instance segmentation with decoder: 100%|██████████| 4/4 [00:00<00:00,  7.51it/s]
Compute Image Embeddings 2D tiled: 100%|██████████| 4/4 [00:01<00:00,  2.37it/s]
Initialize tiled instance segmentation with decoder: 100%|██████████| 4/4 [00:00<00:00,  7.45it/s]
Compute Image Embeddings 2D tiled: 100%|██████████| 4/4 [00:01<00:00,  2.36it/s]
Initialize tiled instance segmentation with decoder: 100%|██████████| 4/4 [00:00<00:00,  7.47it/s]
Compute Image Embeddings 2D tiled: 100%|██████████| 4/4 [00:01<00:00,  2.36it/s]
Initialize tiled instance segmentation with decoder: 100%|██████████| 4/4 [00:00<00:00,  7.38it/s]
Compute Image Embeddings 2D tiled: 100%|██████████| 4/4 [00:01<00:00,  2.36it/s]
Initialize tiled instance segmentation with decoder: 100%|██████████| 4/4 [00:00<00:00,  7.42it/s]
Compute Image Embeddings 2D tiled: 100%|██████████| 4/4 [00:01<00:00,  2.11it/s]
Initialize tiled in

Stacking inference results...
Prediction shape: (75, 1101, 1101)
Inference complete. Output from this cell will be used for label connection.


In [10]:
# ============================
# 5. 3D CONNECTED COMPONENT LABELING (LABEL STITCHING)
# ============================
# Use the predictions_3d from previous cell
labels_3d = predictions_3d.astype(np.uint32)
print(f"Original unique labels: {len(np.unique(labels_3d)) - 1}")  # -1 for background

# Create binary mask (object vs background)
binary_mask = labels_3d > 0

# Use 3D connected component labeling with 26-connectivity
# This assigns the SAME ID to all connected voxels, even if they had different IDs before
structure_3d = generate_binary_structure(3, 3)  # 26-connectivity (full 3D neighborhood)
stitched_labels, num_features = ndimage_label(binary_mask, structure=structure_3d)

print(f"After 3D connected component labeling: {num_features} connected objects")
print(f"Label stitching complete. Output will be used for blob removal.")

Original unique labels: 10
After 3D connected component labeling: 63 connected objects
Label stitching complete. Output will be used for blob removal.


In [11]:
# ============================
# 6. BLOB REMOVAL & FINAL SAVE
# ============================
print("Applying blob removal filter...")

# Use the stitched_labels from previous cell
labels_for_cleaning = stitched_labels.copy()

print(f"Object count before blob removal: {len(np.unique(labels_for_cleaning)) - 1}")

# Remove all objects smaller than threshold
# min_size means "keep everything this size or bigger"
clean_labels = remove_small_objects(
    labels_for_cleaning, 
    min_size=BLOB_REMOVAL_THRESHOLD, 
    connectivity=1
)

new_count = len(np.unique(clean_labels)) - 1
removed_count = len(np.unique(labels_for_cleaning)) - 1 - new_count
print(f"Object count after blob removal: {new_count}")
print(f"Removed {removed_count} blobs (threshold: {BLOB_REMOVAL_THRESHOLD} voxels)")

# ============================
# 7. SAVE FINAL OUTPUT
# ============================
print(f"Saving final result to: {OUTPUT_MASK_PATH}")

# Create directory if it doesn't exist
os.makedirs(os.path.dirname(OUTPUT_MASK_PATH), exist_ok=True)

# Check if we need 32-bit (if labels > 65k)
dtype = np.uint32 if new_count > 65535 else np.uint16
tifffile.imwrite(OUTPUT_MASK_PATH, clean_labels.astype(dtype))

print(f"✓ Pipeline complete!")
print(f"✓ Final output shape: {clean_labels.shape}")
print(f"✓ Final object count: {new_count}")
print(f"✓ Saved to: {OUTPUT_MASK_PATH}")

Applying blob removal filter...
Object count before blob removal: 63
Object count after blob removal: 20
Removed 43 blobs (threshold: 1450 voxels)
Saving final result to: /home/students/rawat/czi_output/Annotations/Processed_labels/pred_test/PN_KC_488_568_Brain1_CalyxL_LSMPlus_16bit_mask_5_labels_896_tiling.tif
✓ Pipeline complete!
✓ Final output shape: (75, 1101, 1101)
✓ Final object count: 20
✓ Saved to: /home/students/rawat/czi_output/Annotations/Processed_labels/pred_test/PN_KC_488_568_Brain1_CalyxL_LSMPlus_16bit_mask_5_labels_896_tiling.tif
